In [1]:
import cv2
import numpy as np
import threading
import pyttsx3
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.preprocessing.text import Tokenizer

In [2]:
# Load the trained caption model
caption_model = load_model("model.keras")

# Load DenseNet201 feature extractor
feature_extractor = DenseNet201(include_top=False, weights="imagenet", pooling="avg")

# Define Tokenizer and max_length
tokenizer = Tokenizer()
max_length = 36  # 🔥 Must match training

# Initialize text-to-speech engine
tts_engine = pyttsx3.init()
tts_engine.setProperty("rate", 160)  # Adjust speech speed

In [3]:

# Function to preprocess the image
def preprocess_frame(frame, img_size=128):
    frame = cv2.resize(frame, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    frame = img_to_array(frame) / 255.0
    frame = np.expand_dims(frame, axis=0)
    return frame

# Function to predict caption
def predict_caption(model, frame):
    image_feature = feature_extractor.predict(frame, verbose=0)

    in_text = "startseq"
    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)

        y_pred = model.predict([image_feature, sequence], verbose=0)
        y_pred = np.argmax(y_pred)

        word = next((w for w, i in tokenizer.word_index.items() if i == y_pred), None)
        if word is None or word == "endseq":
            break
        in_text += " " + word

    caption = in_text.replace("startseq", "").strip()
    print(f"[DEBUG] Generated Caption: {caption}")  # Debugging
    return caption



In [4]:
# Initialize OpenCV Video Capture
cap = cv2.VideoCapture(0)
caption_text = "Analyzing..."

def update_caption():
    global caption_text
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        processed_frame = preprocess_frame(frame)
        caption_text = predict_caption(caption_model, processed_frame)

        # Speak the caption
        tts_engine.say(caption_text)
        tts_engine.runAndWait()

# Start caption generation in a separate thread
thread = threading.Thread(target=update_caption, daemon=True)
thread.start()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Overlay text
    cv2.putText(frame, caption_text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.imshow("Real-Time Image Captioning", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Generated Caption: 
[DEBUG] Image Feature Shape: (1, 1920)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Seq

In [5]:
import cv2
import numpy as np
import threading
import pyttsx3
import pickle
import queue
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications import DenseNet201

# Load caption model
caption_model = load_model("model.keras")

# Load DenseNet201
feature_extractor = DenseNet201(include_top=False, weights="imagenet", pooling="avg")

# Load tokenizer
try:
    with open("tokenizer.pkl", "rb") as f:
        tokenizer = pickle.load(f)
    print("[INFO] Tokenizer loaded successfully.")
except FileNotFoundError:
    print("[ERROR] Tokenizer file not found!")
    exit()

max_length = 36
caption_text = "Analyzing..."

# Text-to-speech setup
tts_engine = pyttsx3.init()
tts_engine.setProperty("rate", 160)
speech_queue = queue.Queue()

def tts_worker():
    while True:
        text = speech_queue.get()
        if text is None:
            break
        tts_engine.say(text)
        tts_engine.runAndWait()
        speech_queue.task_done()

# Start TTS worker thread
tts_thread = threading.Thread(target=tts_worker, daemon=True)
tts_thread.start()

# Preprocess image
def preprocess_frame(frame, img_size=224):
    frame = cv2.resize(frame, (img_size, img_size))
    frame = img_to_array(frame) / 255.0
    return np.expand_dims(frame, axis=0)

# Predict caption
def predict_caption(model, frame):
    image_feature = feature_extractor.predict(frame, verbose=0)
    print(f"[DEBUG] Image Feature Shape: {image_feature.shape}")
    in_text = "startseq"
    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)
        print(f"[DEBUG] Input Sequence Shape: {sequence.shape}")
        y_pred = model.predict([image_feature, sequence], verbose=0)
        y_pred = np.argmax(y_pred)
        word = next((w for w, i in tokenizer.word_index.items() if i == y_pred), None)
        if word is None or word == "endseq":
            break
        in_text += " " + word
    caption = in_text.replace("startseq", "").strip()
    print(f"[DEBUG] Generated Caption: {caption}")
    return caption if caption else "No caption generated"

# Open webcam
cap = cv2.VideoCapture(0)

def update_caption():
    global caption_text
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        processed_frame = preprocess_frame(frame)
        caption_text = predict_caption(caption_model, processed_frame)
        speech_queue.put(caption_text)

# Start caption thread
threading.Thread(target=update_caption, daemon=True).start()

# Main video loop
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    cv2.putText(frame, caption_text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1,
                (255, 255, 255), 2, cv2.LINE_AA)
    cv2.imshow("Real-Time Image Captioning", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        speech_queue.put(None)  # signal TTS to stop
        break

cap.release()
cv2.destroyAllWindows()


[INFO] Tokenizer loaded successfully.
[DEBUG] Image Feature Shape: (1, 1920)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)


Exception in thread Thread-6 (tts_worker):
Traceback (most recent call last):
  File "C:\Python312\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "C:\Users\subra\AppData\Roaming\Python\Python312\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Python312\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\subra\AppData\Local\Temp\ipykernel_25236\170191154.py", line 41, in tts_worker
  File "C:\Users\subra\AppData\Roaming\Python\Python312\site-packages\pyttsx3\engine.py", line 180, in runAndWait
    raise RuntimeError('run loop already started')
RuntimeError: run loop already started


[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Generated Caption: man in blue down is sitting on the object
[DEBUG] Image Feature Shape: (1, 1920)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Generated Caption: two people are sitting on the object
[DEBUG] Image Feature Shape: (1, 1920)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: (1, 36)
[DEBUG] Input Sequence Shape: